# 06 - Task 7: pairwise judging with Kaggle models
Create this notebook as a **utility benchmark** notebook so `kaggle_benchmarks` model-proxy calls are available. Attach outputs of notebooks 04 and 05. No GPU needed. Judging only reads saved pair files.

In [ ]:
# Get the code: preferably from the private Kaggle dataset "advanced-genai-pa2-code"
# (attached as input); otherwise clone the private GitHub repo using a Kaggle secret
# named GITHUB_TOKEN (Add-ons -> Secrets).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import glob, os, shutil, subprocess, sys
if not os.path.exists(REPO):
    code = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
            if "advanced-genai-pa2-code" in f]
    if code:
        shutil.copytree(code[0], REPO)
        print("code copied from", code[0])
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
        del token, authed
os.chdir(REPO)
print(open("CODE_VERSION.txt").read() if os.path.exists("CODE_VERSION.txt") else
      subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import glob, shutil
import kaggle_benchmarks as kbench
print(list(kbench.llms.keys()) if hasattr(kbench.llms, "keys") else kbench.llms)

In [ ]:
# Copy saved pair files (and any earlier judgments, so reruns resume) into the repo.
os.makedirs("outputs/eval", exist_ok=True)
for f in glob.glob("/kaggle/input/**/outputs/eval/*.jsonl", recursive=True):
    dst = os.path.join("outputs/eval", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
!ls -l outputs/eval

In [ ]:
JUDGE = "google/gemini-3.6-flash"   # record the model actually used in the report
# (pairs file, judged-file name, criteria override)
JOBS = [("uf_dpo_vs_sft", "uf_dpo_vs_sft", None),
        ("hh_dpo_vs_sft", "hh_dpo_vs_sft", None),
        ("persona_lora_vs_instruct", "persona_lora_vs_instruct", None),               # style + useful + safe
        ("persona_lora_vs_instruct", "persona_lora_vs_instruct_style", "persona_style"),  # style only
        ("persona_lora_vs_instruct", "persona_lora_vs_instruct_helpful", "helpful_safe"), # usefulness/safety only
        ("sft_vs_base_alpaca", "sft_vs_base_alpaca", None),
        ("xhh_dpo_vs_sft_alpaca", "xhh_dpo_vs_sft_alpaca", None),
        ("xuf_dpo_vs_sft_hh", "xuf_dpo_vs_sft_hh", None),
        # documented deviations (section 14.3): stronger HH-DPO and LoRA v2
        ("hh_dpo_lr5e-6_vs_sft", "hh_dpo_lr5e-6_vs_sft", None),
        ("xhh_dpo_lr5e-6_vs_sft_alpaca", "xhh_dpo_lr5e-6_vs_sft_alpaca", None),
        ("persona_lora_v2_vs_instruct", "persona_lora_v2_vs_instruct", None),
        ("persona_lora_v2_vs_instruct", "persona_lora_v2_vs_instruct_style", "persona_style"),
        ("persona_lora_v2_vs_instruct", "persona_lora_v2_vs_instruct_helpful", "helpful_safe")]
for pairs, out, kind in JOBS:
    if not os.path.exists(f"outputs/eval/{pairs}_pairs.jsonl"):
        print("missing", pairs); continue
    kind_arg = f"--eval_kind {kind}" if kind else ""
    !python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{pairs}_pairs.jsonl         --output_path outputs/eval/{out}_judged.jsonl --judge_model {JUDGE} {kind_arg}         --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
    !python scripts/analyze_results.py --judged outputs/eval/{out}_judged.jsonl --pairs outputs/eval/{pairs}_pairs.jsonl --out report/analysis/{out}.md

In [ ]:
!python scripts/summarize_results.py --eval_dir outputs/eval --out report/results_table.md
!cat report/results_table.md

In [ ]:
# Small archive of metrics/configs/generations (model weights are already kept as notebook output;
# zipping them too would double disk use).
!cd {REPO} && zip -qr /kaggle/working/judge_outputs.zip outputs/eval report/results_table.md report/analysis -x "*.safetensors" "*.bin" "*/trainer/*"
!ls -lh /kaggle/working/*.zip